# Local CPU Component Test (<= 30 minutes)

Этот ноутбук предназначен для **локального** smoke/e2e теста MVP-компонентов на CPU:

1. Создание маленького subset из YOLO-датасета.
2. Валидация структуры subset.
3. Анализ датасета (stats JSON/CSV).
4. Генерация adaptive policy и decision report.
5. Инференс на val subset с уже обученной YOLO.
6. YOLO->COCO конвертация и COCOeval.

Ожидаемая длительность при дефолтных параметрах: обычно 10-30 минут на CPU.

In [1]:
from __future__ import annotations

import json
import sys
import time
from pathlib import Path

# Автоопределяем корень проекта.
# Если ноутбук запущен из project_root -> берем cwd.
# Если из notebooks/ -> берем parent.
cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd if (cwd / 'src').exists() else cwd.parent
assert (PROJECT_ROOT / 'src').exists(), f'Cannot find project root from {cwd}'
sys.path.insert(0, str(PROJECT_ROOT))

print('PROJECT_ROOT =', PROJECT_ROOT)

AssertionError: Cannot find project root from /content

In [ ]:
from src.utils.io import load_yaml, dump_json
from src.data.subset_builder import build_yolo_subset
from src.data.visdrone_manager import validate_visdrone_yolo_structure, save_validation_report
from src.analysis.dataset_analyzer import DatasetAnalyzerConfig, analyze_dataset
from src.policy.rule_engine import RuleEngineConfig, generate_policy_from_stats, save_policy_artifacts
from src.evaluation.coco_converter import convert_yolo_gt_to_coco, convert_yolo_pred_txt_to_coco
from src.evaluation.coco_eval_runner import run_coco_eval
from src.evaluation.metrics_report import build_markdown_report

project_cfg = load_yaml(PROJECT_ROOT / 'configs' / 'project_config.yaml')

LOCAL_TEST_CFG = {
    # Ограничиваем размер subset для CPU-лимита времени.
    'train_subset_size': 120,
    'val_subset_size': 40,
    'seed': 42,
    'imgsz_infer': 640,
    'max_wall_time_sec': 1800,  # 30 минут
    'predict_conf': 0.25,
    'predict_iou': 0.7,
}

# ВАЖНО: укажи путь к уже обученным весам YOLO.
# Пример: PROJECT_ROOT / 'runs' / 'my_exp' / 'weights' / 'best.pt'
PRETRAINED_WEIGHTS = PROJECT_ROOT / 'weights' / 'best.pt'

dataset_root = Path(project_cfg['dataset']['root'])
if not dataset_root.is_absolute():
    dataset_root = PROJECT_ROOT / dataset_root

subset_root = PROJECT_ROOT / 'artifacts' / 'local_cpu_subset'
local_stats_dir = PROJECT_ROOT / 'artifacts' / 'local_cpu_stats'
local_policy_dir = PROJECT_ROOT / 'artifacts' / 'local_cpu_policy'
local_eval_dir = PROJECT_ROOT / 'artifacts' / 'local_cpu_eval'
local_runs_dir = PROJECT_ROOT / 'runs' / 'local_cpu_component_test'
for p in [local_stats_dir, local_policy_dir, local_eval_dir, local_runs_dir]:
    p.mkdir(parents=True, exist_ok=True)

print('Dataset root:', dataset_root)
print('Subset root:', subset_root)
print('Weights path:', PRETRAINED_WEIGHTS)

In [ ]:
# Utility для замеров времени по этапам.
stage_times = {}
wall_start = time.perf_counter()

class StageTimer:
    def __init__(self, name):
        self.name = name
        self.start = None

    def __enter__(self):
        self.start = time.perf_counter()
        print(f'[{self.name}] started')
        return self

    def __exit__(self, exc_type, exc, tb):
        elapsed = time.perf_counter() - self.start
        stage_times[self.name] = elapsed
        print(f'[{self.name}] finished in {elapsed:.1f} sec')

In [ ]:
# 1) Строим маленький subset для ускоренного CPU-теста.
with StageTimer('subset_build'):
    subset_report = build_yolo_subset(
        dataset_root=dataset_root,
        output_root=subset_root,
        train_images=LOCAL_TEST_CFG['train_subset_size'],
        val_images=LOCAL_TEST_CFG['val_subset_size'],
        seed=LOCAL_TEST_CFG['seed'],
        image_extensions=tuple(project_cfg['dataset'].get('image_extensions', ['.jpg', '.jpeg', '.png', '.bmp'])),
        clean_output=True,
    )

print(subset_report)
assert subset_report.train_images > 0 and subset_report.val_images > 0, 'Subset is empty'

In [ ]:
# 2) Валидация subset и 3) анализ признаков.
with StageTimer('validation'):
    validation = validate_visdrone_yolo_structure(
        dataset_root=subset_root,
        splits=('train', 'val'),
        image_extensions=tuple(project_cfg['dataset'].get('image_extensions', ['.jpg', '.jpeg', '.png', '.bmp'])),
    )
    save_validation_report(validation, local_stats_dir / 'validation_report.json')
    print('validation.is_valid =', validation['is_valid'])

with StageTimer('analysis'):
    analyzer_cfg = DatasetAnalyzerConfig(
        small_max_area=float(project_cfg['analysis']['small_max_area']),
        medium_max_area=float(project_cfg['analysis']['medium_max_area']),
        tiny_max_area=float(project_cfg['analysis']['tiny_max_area']),
        image_extensions=tuple(project_cfg['dataset'].get('image_extensions', ['.jpg', '.jpeg', '.png', '.bmp'])),
        generate_plots=False,  # для локального быстрого прогона отключаем графики
    )
    stats = analyze_dataset(
        dataset_root=subset_root,
        output_dir=local_stats_dir,
        splits=('train', 'val'),
        config=analyzer_cfg,
    )

print('stats files:', local_stats_dir / 'dataset_stats.json', local_stats_dir / 'dataset_stats.csv')

In [ ]:
# 4) Генерируем adaptive policy + explainable report.
with StageTimer('policy_generation'):
    rule_cfg = RuleEngineConfig.from_project_config(project_cfg)
    policy, decision_report = generate_policy_from_stats(stats, cfg=rule_cfg)
    paths = save_policy_artifacts(policy=policy, decision_report=decision_report, output_dir=local_policy_dir)

print('policy saved:', paths)
print('fired rules:', len(decision_report.get('fired_rules', [])))

In [ ]:
# 5) Инференс на CPU уже обученной моделью YOLO.
# Чтобы уложиться в время, используем только val subset.
assert PRETRAINED_WEIGHTS.exists(), (
    f'Не найден файл весов: {PRETRAINED_WEIGHTS}. '\
    'Укажи путь к уже обученной YOLO модели.'
)

with StageTimer('yolo_inference_cpu'):
    from ultralytics import YOLO

    model = YOLO(str(PRETRAINED_WEIGHTS))
    predict_results = model.predict(
        source=str(subset_root / 'images' / 'val'),
        imgsz=int(LOCAL_TEST_CFG['imgsz_infer']),
        conf=float(LOCAL_TEST_CFG['predict_conf']),
        iou=float(LOCAL_TEST_CFG['predict_iou']),
        device='cpu',
        save=False,
        save_txt=True,
        save_conf=True,
        project=str(local_runs_dir),
        name='predict',
        exist_ok=True,
        verbose=False,
    )

pred_labels_dir = local_runs_dir / 'predict' / 'labels'
assert pred_labels_dir.exists(), f'Pred labels dir not found: {pred_labels_dir}'
print('pred labels:', pred_labels_dir)
print('predicted images:', len(predict_results))

In [ ]:
# 6) Конвертация в COCO и запуск COCOeval.
with StageTimer('coco_eval'):
    coco_gt_path = local_eval_dir / 'coco_gt.json'
    coco_dt_path = local_eval_dir / 'coco_dt.json'
    coco_eval_path = local_eval_dir / 'coco_eval.json'

    convert_yolo_gt_to_coco(
        images_dir=subset_root / 'images' / 'val',
        labels_dir=subset_root / 'labels' / 'val',
        class_names=project_cfg['dataset']['visdrone_classes'],
        output_path=coco_gt_path,
        image_extensions=tuple(project_cfg['dataset'].get('image_extensions', ['.jpg', '.jpeg', '.png', '.bmp'])),
    )
    convert_yolo_pred_txt_to_coco(
        pred_labels_dir=pred_labels_dir,
        images_dir=subset_root / 'images' / 'val',
        output_path=coco_dt_path,
        image_extensions=tuple(project_cfg['dataset'].get('image_extensions', ['.jpg', '.jpeg', '.png', '.bmp'])),
    )
    metrics = run_coco_eval(
        coco_gt_path=coco_gt_path,
        coco_dt_path=coco_dt_path,
        output_path=coco_eval_path,
        use_tiny_eval=True,
        tiny_max_area=float(project_cfg['analysis']['tiny_max_area']),
    )
    report_md = build_markdown_report(
        metrics_by_run={'local_cpu_pretrained_eval': metrics},
        output_path=PROJECT_ROOT / 'artifacts' / 'reports' / 'local_cpu_component_report.md',
    )

print(json.dumps(metrics, indent=2, ensure_ascii=False))
print('report:', report_md)

In [ ]:
# Финальный тайминг и контроль лимита 30 минут.
wall_total = time.perf_counter() - wall_start
stage_times_sorted = dict(sorted(stage_times.items(), key=lambda kv: kv[1], reverse=True))

summary = {
    'total_wall_time_sec': round(wall_total, 2),
    'max_allowed_sec': LOCAL_TEST_CFG['max_wall_time_sec'],
    'within_30_min': wall_total <= LOCAL_TEST_CFG['max_wall_time_sec'],
    'stage_times_sec': {k: round(v, 2) for k, v in stage_times_sorted.items()},
}
dump_json(summary, PROJECT_ROOT / 'artifacts' / 'reports' / 'local_cpu_timing_summary.json')

print(json.dumps(summary, indent=2, ensure_ascii=False))
if not summary['within_30_min']:
    print('\nWARNING: превышен лимит 30 минут. Уменьши subset (train/val) или imgsz_infer.')